# Projeto: Testes – Coleta de Previsão Climática (ClimaTempo)

### 📌 Objetivo
Este notebook tem como finalidade validar, por meio de testes unitários, as funções desenvolvidas para o projeto de coleta de dados meteorológicos do site [ClimaTempo](https://www.climatempo.com.br/previsao-do-tempo).

Todos os testes são realizados utilizando PySpark e `pytest` com `assert`, diretamente no ambiente Databricks, assegurando que as funções de transformação, extração e tratamento estejam funcionando conforme esperado.

---

### 🧪 Funções Testadas

- `salvar_no_datalake()`
- `adicionaColunasDeCarga()`
- `inserir_com_merge()`
- `consultar_previsao_api()`
- `extrair_dados_previsao()`
- `previsao_do_tempo()`
- `ler_tabelas()`
- `alterar_tipos()`
- `renomear_colunas()`
- `tratar_celulas()`
- `junta_uf_e_cidade()`
- `extrair_texto()`
- `obter_previsao_do_dia()`
- `addDays()`
- `coletar_previsoes())`
- `montar_url_tempo()`
- `extrair_dados()`
- `obter_previsoes_todas_capitais_em_df()`
- `get_html_from_url()`
- `aggregate_forecast()`
- `generate_spark_df()`
- `coletar_previsoes_refatorado()`
---

### ⚙️ Tecnologias e Ferramentas

- **PySpark**
- **Pytest**
- **Python**
- **Databricks Notebooks**
- **WeatherAPI & ClimaTempo (como fontes de dados)**

---

### ✅ Boas Práticas Adotadas

- Testes com diferentes tipos de entrada: válidos, inválidos e nulos
- Comparações via `assert` simples no PySpark (sem pandas)
- Separação clara entre Arrange, Act e Assert
- Verificações de comportamento esperado em tipos, valores e colunas
- Compatibilidade com execução manual e automática no Databricks

---

### 📅 Histórico de Atualizações

| Data       | Responsável   | Versão | Descrição                                                           |
|------------|---------------|--------|---------------------------------------------------------------------|
| 16/05/2025 | Lucas Zorn    | 1.0    | Criação do arquivo de testes e execução dos testes iniciais         |
| 17/05/2025 | Lucas Zorn    | 1.1    | Documentação e conclusão da estrutura de testes unitários           |

---


# Testes unitários das funções

In [0]:
%pip install pytest
import pytest

In [0]:
%run ../../config/paths

In [0]:
%run ../utils/funcoes_coleta_dados/funcoes_coleta_dados_run

In [0]:
%run ../utils/utils_run_gold

In [0]:
%run ../utils/utils_run_silver

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS grupo01.testes

## 1.0 Criando um dataframe apenas para testar as funções

In [0]:
dados = [
    ("João Silva", "23,8", "82,286121", "1", "São Paulo", "1", "2", "Masculino"),
    ("Maria Souza", "31,2", "56,734567", "0", "Rio de Janeiro", "1", None, "Feminino"),
    ("João Silva", "23,8", "82,286121", "1", "São Paulo", "1", "2", "Masculino"),
    ("Ana Clara", "29,5", None, "1", "Belo Horizonte", "0", "0", None),
    ("Carlos Lima", None, "39,50", "0", None, "0", "1", "Masculino"),
    ("Carlos Lima", "45,1", "120,900001", "0", "Curitiba", "0", "0", "Masculino")
]

colunas = [
    "Nome Completo",
    "Idade ",
    "Salário Bruto",
    "Plano de Saúde",
    "Cidade de Origem",
    "Tem dependentes",
    "Número de Filhos",
    "Gênero "
]

df = spark.createDataFrame(dados, colunas)

## 2.0 Testes

### 2.1 Teste "salvar_no_datalake"

In [0]:
def test_salvar_no_datalake():
    # Arrange
    dados_teste = [("João", "23,8"), ("Maria", "31,2")]
    colunas = ["nome", "idade"]
    df = spark.createDataFrame(dados_teste, colunas)
    caminho_teste = landingzone_test

    # Remove o diretório antes do teste (garante modo overwrite limpo)
    dbutils.fs.rm(caminho_teste, True)

    # Act
    salvar_no_datalake(
        dataframe=df,
        path=caminho_teste,
        modo="overwrite",
        format="delta",
        data_carga=False
    )

    # Assert
    df_lido = spark.read.format("delta").load(caminho_teste)
    resultado = df_lido.collect()

    assert len(resultado) == 2
    assert resultado[0]["nome"] in ["João", "Maria"]
    assert "idade" in df_lido.columns

In [0]:
test_salvar_no_datalake()

### 2.2 Teste "adicionaColunasDeCarga"

In [0]:
def test_adicionaColunasDeCarga():
    # Arrange
    dados_teste = [("João", "23,8"), ("Maria", "31,2")]
    colunas = ["nome", "idade"]
    df = spark.createDataFrame(dados_teste, colunas)

    # Act
    df_com_carga = adicionaColunasDeCarga(df)

    # Assert
    assert "data_carga" in df_com_carga.columns

    resultado = df_com_carga.select("data_carga").collect()
    assert resultado[0]["data_carga"] is not None
    assert len(resultado[0]["data_carga"]) == 19  # formato: 'yyyy-MM-dd HH:mm:ss'


In [0]:
test_adicionaColunasDeCarga()

### 2.3 Teste "inserir_com_merge"

In [0]:
from delta.tables import DeltaTable

def test_inserir_com_merge():
    # Arrange - cria dados de origem
    dados_iniciais = [("João", "23,8"), ("Maria", "31,2")]
    colunas = ["nome", "idade"]
    df_origem = spark.createDataFrame(dados_iniciais, colunas)

    # Define tabela
    tabela_teste = "grupo01.testes.tabela_teste"

    # Garante remoção da tabela para começar teste limpo
    spark.sql(f"DROP TABLE IF EXISTS {tabela_teste}")

    # Act - 1ª execução cria a tabela
    inserir_com_merge(df_origem, tabela_teste, camposChave=["nome"])

    # Modifica o dado da Maria (simula update)
    dados_update = [("Maria", "32,0"), ("Pedro", "28,4")]  # Maria será atualizada, Pedro será inserido
    df_update = spark.createDataFrame(dados_update, colunas)
    inserir_com_merge(df_update, tabela_teste, camposChave=["nome"])

    # Assert - lê da tabela e verifica merge
    df_resultado = spark.table(tabela_teste).select("nome", "idade").orderBy("nome")
    resultado = df_resultado.collect()

    nomes = [r["nome"] for r in resultado]
    idades = [r["idade"] for r in resultado]

    assert nomes == ["João", "Maria", "Pedro"]
    assert "32,0" in idades
    assert len(resultado) == 3


In [0]:
test_inserir_com_merge()

### 2.4 Teste consulta API

### 2.4.1 Teste "consultar_previsao_api"

In [0]:
def test_consultar_previsao_api():
    # Arrange
    api_token = enum_configs.WEATHER_API_KEY.value
    cidade = "São Paulo"
    dias = 1

    # Act
    resultado = consultar_previsao_api(api_token, cidade, dias)

    # Assert
    assert isinstance(resultado, dict)
    assert "forecast" in resultado
    assert "forecastday" in resultado["forecast"]
    assert len(resultado["forecast"]["forecastday"]) == dias


In [0]:
test_consultar_previsao_api()

### 2.4.2 Teste "extrair_dados_previsao"

In [0]:
def test_extrair_dados_previsao():
    # Arrange
    api_token = enum_configs.WEATHER_API_KEY.value
    estado = "SP"
    cidade = "São Paulo"
    json_dados = consultar_previsao_api(api_token, cidade, dias=1)

    # Act
    resultado = extrair_dados_previsao(json_dados, estado, cidade)

    # Assert
    assert isinstance(resultado, list)
    assert len(resultado) == 1

    previsao = resultado[0]
    assert previsao["estado"] == estado
    assert previsao["cidade"] == cidade
    assert "temperatura_maxima" in previsao
    assert "precipitacao" in previsao
    assert "umidade_maxima" in previsao

In [0]:
test_extrair_dados_previsao()

### 2.4.3 Teste "previsao_do_tempo"

In [0]:
def test_previsao_do_tempo():
    # Arrange
    api_key = enum_configs.WEATHER_API_KEY.value
    cidades_dict = cidades
    cidades_test = dict(list(cidades_dict.items())[:2]) # Pega apenas duas

    # Act
    df_resultado = previsao_do_tempo(api_key, cidades_test, dias_previsao=1)

    # Assert
    assert df_resultado.count() >= 1
    assert "cidade" in df_resultado.columns
    assert "temperatura_maxima" in df_resultado.columns
    assert df_resultado.select("cidade").distinct().count() == len(cidades_test)

In [0]:
test_previsao_do_tempo()

### 2.5 Teste "ler_tabelas"

In [0]:
def test_ler_tabelas():
    # Arrange
    nome_tabela = "grupo01.testes.tabela_teste"

    # Act
    df_lido = ler_tabelas(path=nome_tabela, fromTable=True)

    # Assert
    assert df_lido.count() > 0
    assert "nome" in df_lido.columns
    assert "idade" in df_lido.columns

In [0]:
test_ler_tabelas()

### 2.6 Teste "alterar_tipos"

In [0]:
def test_alterar_tipos():
    # Arrange
    dados = [
        Row(valor_str="123", valor_float="45.67", data_str="2024-05-15")
    ]
    df = spark.createDataFrame(dados)

    # Act - aplicar as conversões
    df_convertido = alterar_tipos(df, ["valor_str"], "int")
    df_convertido = alterar_tipos(df_convertido, ["valor_float"], "double")
    df_convertido = alterar_tipos(df_convertido, ["data_str"], "date")
    df_convertido = alterar_tipos(df_convertido, ["valor_float"], "decimal", precisao=6, escala=2)

    resultado = df_convertido.collect()[0]

    # Assert
    assert isinstance(resultado["valor_str"], int)
    assert resultado["valor_str"] == 123

    assert str(type(resultado["valor_float"])).endswith("Decimal'>")
    assert float(resultado["valor_float"]) == 45.67

    assert str(resultado["data_str"]) == "2024-05-15"

In [0]:
test_alterar_tipos()

### 2.7 Teste "renomear_colunas"

In [0]:
def test_renomear_colunas():
    # Arrange
    dados = [("João", 30), ("Maria", 28)]
    colunas_originais = ["nome", "idade"]
    df = spark.createDataFrame(dados, colunas_originais)

    mapeamento = {
        "nome": "nome_completo",
        "idade": "anos",
        "cidade": "municipio"  # coluna inexistente, deve gerar warning
    }

    # Act
    df_renomeado = renomear_colunas(df, mapeamento)

    # Assert
    novas_colunas = df_renomeado.columns
    print(novas_colunas)
    assert "nome_completo" in novas_colunas
    assert "anos" in novas_colunas
    assert "cidade" not in novas_colunas
    assert "nome" not in novas_colunas
    assert "idade" not in novas_colunas


In [0]:
test_renomear_colunas()

### 2.8 Teste "tratar_celulas"

In [0]:
def test_tratar_celulas_completo():
    # Arrange
    dados = [
        ("12mm - 80%", "32°", "21°", "95%", "50%", "15 km/h"),    # Tudo preenchido
        (None, "inválido", "inválido", "xx%", None, "10 mph"),   # Tudo inválido ou nulo
        ("0mm - 0%", "28°", "18°", "80%", "40%", "12,5")          # Zero e valor com vírgula
    ]
    colunas = ["precipitacao", "temp_max", "temp_min", "umid_max", "umid_min", "vel_vento"]
    df = spark.createDataFrame(dados, colunas)

    # Act
    df_resultado = tratar_celulas(df)
    resultado = df_resultado.collect()

    # Assert linha 1
    r1 = resultado[0]
    assert r1.precipitacao == 12.0
    assert r1.prob_precipitacao == 80.0
    assert r1.temp_max == "32"
    assert r1.temp_min == "21"
    assert r1.umid_max == "95"
    assert r1.umid_min == "50"
    assert round(r1.vel_vento, 2) == 15.0

    # Assert linha 2
    r2 = resultado[1]
    assert r2.precipitacao == None
    assert r2.prob_precipitacao == None
    assert r2.temp_max == None
    assert r2.temp_min == None
    assert r2.umid_max == None
    assert r2.umid_min == None
    assert round(r2.vel_vento, 2) == round(10 * 1.60934, 2)  # mph convertido pra km/h

    # Assert linha 3
    r3 = resultado[2]
    assert r3.precipitacao == None
    assert r3.prob_precipitacao == None
    assert r3.temp_max == "28"
    assert r3.temp_min == "18"
    assert r3.umid_max == "80"
    assert r3.umid_min == "40"
    assert round(r3.vel_vento, 2) == 12.5


In [0]:
test_tratar_celulas_completo()

### 2.9 Teste "junta_uf_e_cidade"

In [0]:
def test_junta_uf_e_cidade():
    # Arrange
    cidade = "Porto Alegre"
    uf = "RS"

    # Act
    resultado = junta_uf_e_cidade(cidade, uf)

    # Assert
    assert resultado == "portoalegre-rs"

In [0]:
test_junta_uf_e_cidade()

### 2.10 Teste "extrair_texto"

In [0]:
def test_extrair_texto():
    #Arrange
    seletor = BeautifulSoup("<p> Olá  <b> mundo </b> ! </p>", "html.parser").p
    seletor_null = None

    #Act
    resultado = extrair_texto(seletor)
    resultado_null = extrair_texto(seletor_null)

    #Assert
    assert resultado == "Olá mundo !"
    assert resultado_null is None

In [0]:
test_extrair_texto()

### 2.11 Teste "obter_previsao_do_dia"

In [0]:
def test_obter_previsao_do_dia():
    #Arrange
    html = """
    <div class="previsao-dia">
        <img alt="Temperatura máxima"><span>32°</span>
        <img alt="Temperatura mínima"><span>21°</span>
        <span class="_margin-l-5">12mm - 80%</span>
        <span class="variables-border-identifier -wind"></span>
        <div class="-gray _flex _margin-l-5">15 km/h</div>
        <img alt="Umidade mínima"><span>50%</span>
        <img alt="Umidade máxima"><span>95%</span>
    </div>
    """

    soup = BeautifulSoup(html, "html.parser")
    secao = soup.select_one(".previsao-dia")

    #Act
    resultado = obter_previsao_do_dia(secao)

    #Assert
    assert resultado["temperatura_maxima"] == "32°"
    assert resultado["temperatura_minima"] == "21°"
    assert resultado["precipitacao"] == "12mm - 80%"
    assert resultado["vento"] == "15 km/h"
    assert resultado["umidade_minima"] == "50%"
    assert resultado["umidade_maxima"] == "95%"


In [0]:
test_obter_previsao_do_dia()

### 2.12 Teste "addDays"

In [0]:
def test_addDays():
    #Teste simples sem mudança de mês
    assert addDays("2024-05-10", 5) == "2024-05-15"

    #Teste com transição de mês
    assert addDays("2024-01-30", 3) == "2024-02-02"

    #Teste com dia 0
    assert addDays("2024-12-01", 0) == "2024-12-01"

    #Teste com dia negativo
    assert addDays("2024-06-15", -5) == "2024-06-10"

In [0]:
test_addDays()

### 2.13 Teste "coletar_previsoes"

In [0]:
def test_coletar_previsoes():
    # Executa a função real, com scraping ao site do ClimaTempo
    dados = coletar_previsoes("CAPITAIS_ID_CLIMA_TEMPO")

    # Verificações básicas
    assert isinstance(dados, list), "O retorno deve ser uma lista"
    assert len(dados) > 0, "A lista não deve estar vazia"

    amostra = dados[0]

    # Verificações de campos esperados
    for campo in [
        "temperatura_maxima", "temperatura_minima", "precipitacao", "vento",
        "umidade_maxima", "umidade_minima", "cidade", "estado", "data_previsao"
    ]:
        assert campo in amostra, f"Campo esperado '{campo}' não encontrado"
        assert amostra[campo] is not None, f"O campo '{campo}' está vazio"

    print("Teste passou com sucesso.")
    # print(amostra)


In [0]:
test_coletar_previsoes()

### 2.14 Teste "montar_url_tempo"

In [0]:
def test_montar_url_tempo():
    #Arrange, Act
    url = montar_url_tempo("Porto Alegre", "RS", "hoje")
    expected = "https://www.cptec.inpe.br/api/hoje?city=Porto%20Alegre%2C%20RS" ##[...]/hoje?city=Porto Alegre, RS | %20 = espaço e %2C = vírgula
    
   #Assert 
    assert url == expected

In [0]:
test_montar_url_tempo()

### 2.15 Teste "extrair_dados"

In [0]:
def test_extrair_dados():
    dados_mock = {
        "previsao": [
            {"data_atual": "2025-05-17", "temperatura_max": 30, "temperatura_min": 20, "probabilidade_chuva": 50},
            {"data_atual": "2025-05-18", "temperatura_max": 31, "temperatura_min": 21, "probabilidade_chuva": 40}
        ]
    }

    resultado = extrair_dados(dados_mock, tipo="proximos_dias")

    assert resultado[0]["data_previsao"] == "2025-05-17"
    assert resultado[0]["temperatura_maxima"] == "30"
    assert resultado[0]["temperatura_minima"] == "20"
    assert resultado[0]["precipitacao"] == "50%"
    
    assert resultado[1]["data_previsao"] == "2025-05-18"
    assert resultado[1]["temperatura_maxima"] == "31"
    assert resultado[1]["temperatura_minima"] == "21"
    assert resultado[1]["precipitacao"] == "40%"


In [0]:
test_extrair_dados()

### 2.16 Teste "obter_previsoes_todas_capitais_em_df"

In [0]:
def test_obter_previsoes_todas_capitais_em_df():
    registros = obter_previsoes_todas_capitais_em_df("CAPITAIS_ID_CLIMA_TEMPO")[:20]
    

    assert isinstance(registros, list), "A saída deve ser uma lista"
    assert len(registros) > 0, "A lista de registros não deve estar vazia"

    amostra = registros[0]

    # Verifica se os campos esperados existem na amostra
    campos_esperados = [
        "data_previsao",
        "temperatura_maxima",
        "temperatura_minima",
        "precipitacao",
        "fonte",
        "estado",
        "cidade"
    ]

    for campo in campos_esperados:
        assert campo in amostra, f"Campo ausente: {campo}"
        assert amostra[campo] is not None, f"Campo '{campo}' está com valor None"

In [0]:
test_obter_previsoes_todas_capitais_em_df()

### 2.17 Teste "get_html_from_url"

In [0]:
def test_get_html_from_url():
    # Exemplo de cidade_id real — Porto Alegre, RS
    cidade_id = "BRXX0503:1:BR"  # Esse é um exemplo comum usado pela weather.com

    html = get_html_from_url(cidade_id)

    # Verificações básicas
    assert isinstance(html, str), "O retorno deve ser uma string"
    assert "<html" in html.lower(), "A resposta deve conter estrutura HTML"
    assert "clima" in html.lower() or "tempo" in html.lower(), "A página deve conter conteúdo relacionado a clima"


In [0]:
test_get_html_from_url()

### 2.18 Teste "parse_forecast_data_real"

In [0]:
def test_parse_forecast_data_real():
    #Arrange
    html = get_html_from_url("BRXX0503:1:BR")
    data_atual = datetime.now().strftime("%Y-%m-%d")

    #Act
    resultado = parse_forecast_data(html, data_atual)

    #Assert
    assert isinstance(resultado, list), "Resultado deve ser uma lista"
    assert len(resultado) > 0, "Lista de previsões deve ter ao menos 1 item"

    exemplo = resultado[0]

    for campo in [
        "day", "period", "temperatura_maxima", "temperatura_minima",
        "precipitacao", "vento", "umidade_maxima", "umidade_minima"
    ]:
        assert campo in exemplo, f"Campo ausente: {campo}"

In [0]:
test_parse_forecast_data_real()

### 2.19 Teste "aggregate_forecast"

In [0]:
def test_aggregate_forecast():
    #Arrange
    registros = [
        {
            "day": "2025-05-17",
            "period": "Dia",
            "temperatura_maxima": "31°",
            "temperatura_minima": "",
            "precipitacao": "60%",
            "vento": "SE 22 km/h",
            "umidade_maxima": "",
            "umidade_minima": "48%"
        },
        {
            "day": "2025-05-17",
            "period": "Noite",
            "temperatura_maxima": "",
            "temperatura_minima": "19°",
            "precipitacao": "",
            "vento": "SE 22 km/h",
            "umidade_maxima": "85%",
            "umidade_minima": ""
        },
        {
            "day": "2025-05-18",
            "period": "Dia",
            "temperatura_maxima": "32°",
            "temperatura_minima": "",
            "precipitacao": "40%",
            "vento": "S 18 km/h",
            "umidade_maxima": "",
            "umidade_minima": "42%"
        },
        {
            "day": "2025-05-18",
            "period": "Noite",
            "temperatura_maxima": "",
            "temperatura_minima": "20°",
            "precipitacao": "",
            "vento": "S 18 km/h",
            "umidade_maxima": "80%",
            "umidade_minima": ""
        }
    ]

    #Act
    resultado = aggregate_forecast(registros)

    #Assert
    assert isinstance(resultado, list)
    assert len(resultado) == 2

    dia1 = resultado[0]
    assert dia1["data_previsao"] == "2025-05-17"
    assert dia1["temperatura_maxima"] == "31°"
    assert dia1["temperatura_minima"] == "19°"
    assert dia1["precipitacao"] == "60%"
    assert dia1["vento"] == "SE 22 km/h"
    assert dia1["umidade_maxima"] == "85%"
    assert dia1["umidade_minima"] == "48%"

    dia2 = resultado[1]
    assert dia2["data_previsao"] == "2025-05-18"
    assert dia2["temperatura_maxima"] == "32°"
    assert dia2["temperatura_minima"] == "20°"
    assert dia2["precipitacao"] == "40%"
    assert dia2["vento"] == "S 18 km/h"
    assert dia2["umidade_maxima"] == "80%"
    assert dia2["umidade_minima"] == "42%"

In [0]:
test_aggregate_forecast()

### 2.20 Teste "generate_spark_df"

In [0]:
def test_generate_spark_df():
    #Arrange
    dados_mock = [
        {
            "data_previsao": "2025-05-17",
            "temperatura_maxima": "30°",
            "temperatura_minima": "20°",
            "precipitacao": "40%",
            "vento": "NE 22 km/h",
            "umidade_maxima": "85%",
            "umidade_minima": "48%",
            "estado": "RS",
            "cidade": "Porto Alegre"
        },
        {
            "data_previsao": "2025-05-18",
            "temperatura_maxima": "31°",
            "temperatura_minima": "21°",
            "precipitacao": "10%",
            "vento": "SE 20 km/h",
            "umidade_maxima": "82%",
            "umidade_minima": "44%",
            "estado": "RS",
            "cidade": "Porto Alegre"
        }
    ]

    #Act
    df = generate_spark_df(dados_mock)

    #Assert
    assert df.count() == 2, "DataFrame deve ter 2 registros"
    assert set(df.columns) == {
        "data_previsao", "temperatura_maxima", "temperatura_minima",
        "precipitacao", "vento", "umidade_maxima", "umidade_minima",
        "estado", "cidade"
    }, "As colunas do DataFrame não estão corretas"

In [0]:
test_generate_spark_df()

### 2.21 Teste "coletar_previsoes_refatorado"

In [0]:
def test_coletar_previsoes_refatorado():
    #Arrange, Act
    df = coletar_previsoes_refatorado("IDS_CIDADES_WEATHER")
    
    expected_cols = {
        "data_previsao", "temperatura_maxima", "temperatura_minima",
        "precipitacao", "vento", "umidade_maxima", "umidade_minima",
        "estado", "cidade"
    }

    #Assert
    assert df.count() > 0, "O DataFrame retornado está vazio"
    assert set(df.columns) == expected_cols, "As colunas estão incorretas ou incompletas"

In [0]:
test_coletar_previsoes_refatorado()